## Read the interaction data as graph

In [128]:
import os
import sys
from pathlib import Path

# setting proper working directory
PROJECT_DIRECTORY = Path(os.path.abspath('')).resolve()
sys.path.extend([str(PROJECT_DIRECTORY)])

print(f'Python {sys.version} on {sys.platform}')
print('Project directory: ', PROJECT_DIRECTORY)

Python 3.12.7 (main, Oct  1 2024, 02:05:46) [Clang 15.0.0 (clang-1500.3.9.4)] on darwin
Project directory:  /Users/markus/Documents/privat/Studium/Diplomarbeit/RecBole-GNN


In [151]:
import pandas as pd
import numpy as np
import csv

def load_data(filename: str, rows: int = None, dataset: str = "", datatype: str = "sequential") -> pd.DataFrame:

    with open(filename, 'r', encoding="utf-16") as f:
        objects = csv.reader(f, delimiter="\t")

        if rows is None:
            columns_list = list(objects)
        else:
            columns_list = [next(objects) for i in range(rows + 1)]

        if dataset == "MSD":
            df = pd.DataFrame(columns_list, columns=['userID', 'itemID'])
            if any(df.columns != ['userID', 'itemID']):
                df.columns = ['userID', 'itemID']
            df['userID'] = df['userID']
            df['itemID'] = df['itemID']
        elif dataset == "real":
            df = pd.DataFrame(columns_list[1:], columns=columns_list[0])
            if datatype == "sequential":
                df = df.iloc[:, :3]
                if any(df.columns != ["userID", "itemID", "timestamp"]):
                    df.columns = ['userID', 'itemID', 'timestamp']

            elif datatype == "collaborative":
                df = df.iloc[:, :2]
                if any(df.columns != ['userID', 'itemID']):
                    df.columns = ['userID', 'itemID']

            df['userID'] = df['userID'].astype(np.int64)
            df['itemID'] = df['itemID'].astype(np.int64)

    return df

In [177]:
FILENAME_SEQ = PROJECT_DIRECTORY / "asset/data/real-life-raw/user_item_sequence_FILTERED_ANONYMIZED.txt"
DATASET = "real"
DATATYPE = "sequential"
ROWS = 1000000

In [178]:
db_sequential = load_data(FILENAME_SEQ, rows = ROWS, dataset=DATASET, datatype=DATATYPE)
print(db_sequential.shape)
print(db_sequential.keys())

(1000000, 3)
Index(['userID', 'itemID', 'timestamp'], dtype='object')


In [154]:
db_sequential["timestamp"] = db_sequential["timestamp"].apply(lambda x: pd.to_datetime(str(x).split(".")[0], format="%Y-%m-%d %H:%M:%S"))
db_sequential.head()

,userID,itemID,timestamp
0,0,0,2023-02-12 17:06:08
1,0,1,2023-02-12 17:06:57
2,1,2,2023-02-12 17:06:09
3,1,3,2023-02-12 17:06:01
4,1,4,2023-02-12 17:06:55


In [155]:
db_sequential["timestamp"] = db_sequential["timestamp"].apply(lambda x: x.timestamp())
db_sequential.head()

,userID,itemID,timestamp
0,0,0,1.676222e+09
1,0,1,1.676222e+09
2,1,2,1.676222e+09
3,1,3,1.676222e+09
4,1,4,1.676222e+09


In [179]:
db_sequential = db_sequential.drop_duplicates(subset=['userID', 'itemID'])
print(db_sequential.nunique())

userID        15768
itemID        29303
timestamp    725401
dtype: int64


In [180]:
THRESHOLD = 20

#filter_tracks = db_sequential['itemID'].value_counts() > THRESHOLD
#filter_tracks = filter_tracks[filter_tracks].index.tolist()

filter_users = db_sequential['userID'].value_counts() >= THRESHOLD
filter_users = filter_users[filter_users].index.tolist()

#db_sequential = db_sequential[db_sequential['itemID'].isin(filter_tracks)]

db_sequential = db_sequential[db_sequential['userID'].isin(filter_users)]

In [181]:
print(db_sequential['userID'].value_counts().min())
print(db_sequential['itemID'].value_counts().min())

20
1


In [182]:
print(db_sequential.shape)
print(db_sequential.keys())
print(db_sequential.nunique())

(944496, 3)
Index(['userID', 'itemID', 'timestamp'], dtype='object')
userID         9301
itemID        29301
timestamp    695081
dtype: int64


In [183]:
#db_sequential.to_csv("./asset/data/real-life-raw/user_item_sequence_FILTERED_ANONYMIZED.txt", sep="\t", encoding='utf-16', index=False)

In [184]:
db_sequential.columns = ['user_id:token', 'item_id:token', 'timestamp:float']

In [185]:
if ROWS is not None:   
    db_sequential.to_csv(f"./asset/data/real-life-atomic-{ROWS}/real-life-atomic-{ROWS}.inter", sep="\t", encoding='utf-8', index=False)
else:
    db_sequential.to_csv("./asset/data/real-life-atomic/real-life-atomic.inter", sep="\t", encoding='utf-8', index=False)